# What each strategy would make of the dataset

This notebook runs every feature selection strategy written in `src/survey/strategies/` over the features computed on disk and reads the results side by side.
It is a prediction of the resulting dataset, in order to choose the most appropriate approach to the problem.
Consider `notebooks/coverage.ipynb` as a way to perform a qualitative assessment of the strategies, in order to better understand what they would produce if run on the real data.

## Setup

In [ ]:
"""Import the prediction and the tables that read it."""

from sampling.stats import catalogue
from visualization.dataset import progress
from visualization.dataset.plots import reach
from visualization.dataset.tables import coverage, overall, size, tiling

## Sweep the features

If not already co

In [ ]:
"""Read what every strategy makes of every tile of every feature."""

read = progress.read()

## Tiling

A window is searched over one tile, so how wide a strategy cuts its tiles
decides what a window is asked to cover and how much of the record it has to
choose from. Both tables are measured over every tile searched, before any
window is kept or thrown away.


SHARAD is a sounder, so what its record counts as a pixel is a column of a
radargram rather than a picture element, one laid down every 460 m along the
track it flies. Its two columns below count the ones a single pass lays on the
mean tile: straight across it, and the longer way corner to corner.

### How big a tile is

A window is searched over one tile at a time, so the tile is the unit everything
below is measured on. This table says how big a tile the strategy cuts, how much
of a feature really ends up inside one, and how much of an instrument it would
take to cover one.

- **Tile width asked** is `tile_km`, written in the strategy's own YAML file. It
  is a cap, not a size: `tiling.split` picks the fewest even divisions of the
  feature's grid that stay under it, so the real side is at most this wide.
- **Tiles searched** counts every tile the search ran over, across all features.
- **Mean feature ground in a tile**, with its spread, smallest and largest, is
  how much of the *feature* a tile holds. A tile counts only the cells inside
  the feature outline, so one clipping the rim of a crater holds a fraction of
  its own footprint. This is why it reads well below the tile width squared.
- **SHARAD traces across a tile** and **corner to corner**: SHARAD is a sounder,
  so it never fills an area. It lays a single line of traces along the track it
  flies, one every 460 m. These count the traces a single pass leaves crossing a
  tile of the asked width, straight across and the longer way corner to corner.
- **CTX and CRISM pixels to fill a tile** is how many pixels of that instrument
  would cover a whole tile of the asked width, at the median ground one of its
  pixels covers. Read it against the pixels asked in the next table.

The trace and pixel columns are worked out from the tile width asked rather
than from the ground the features really put in a tile, so two strategies
cutting to the same width read the same there.

In [ ]:
"""Tabulate how big a tile each strategy cuts into and what fills one."""

tiling.sizes(read)

### What each instrument lands on a tile

Whether an instrument brings a tile enough to be worth asking it for anything.

- **Mean observations offered** counts the observations of that instrument whose
  footprint lands on a tile at all, over the whole twenty year record and before
  any window is chosen. It is not what a window keeps. A strategy whose tile
  width leaves each feature as a single tile puts every observation of that
  feature onto that one tile, which is why it reads in the hundreds there and in
  the tens once the features are really cut into 100 km tiles.
- **Mean pixels landed** is the pixels the window's own observations leave on a
  tile, added over them, counted as traces for SHARAD.
- **Mean pixels landed per observation** divides those pixels by the
  observations that landed them, so it is what one look at a tile is worth.
- **Pixels asked** is `admits` from the strategy: the pixels an instrument has to
  land on a whole tile before the observation counts as a look at the tile
  rather than a clip of its edge. An observation below it is refused outright.

In [ ]:
"""Tabulate what each instrument lands on a tile and what is asked of it."""

tiling.landed(read)

## Coverage

What a window really leaves on the tile it was picked for: how much of the
tile each instrument reaches, how much of it more than one of them reaches at
once, and how long the window had to run to gather that much. Every tile that
earned a window counts, so an instrument absent from one counts as nothing
there rather than being left out.

### How much of a tile each instrument reaches

What the chosen window actually leaves on the ground, counting a cell once
however often it was revisited. Only tiles that earned a window are counted, and
an instrument absent from one counts as nothing there rather than being left out.

- **Mean of a tile** is the share of the tile the instrument reaches, averaged
  over every kept tile.
- **Spread** is the standard deviation of that share across the tiles.
- **Least** is the worst tile, which should sit at or just above the floor the
  strategy asks for that instrument.
- The **Overlap** row is the share of a tile that *every* instrument reaches at
  once, which is what an observation of the feature would look like if all three
  had to be registered together.

In [ ]:
"""Tabulate how much of a tile each instrument reaches and how much they share."""

coverage.reached(read)

### How long a window runs

- **Mean window**, **Spread** and **Longest** are how far apart the earliest and
  latest observations a kept tile holds are, in days, capped by the strategy's
  `span_days`.
- **Coverage score** is the geometric mean of the shares the window's own
  constraints reach, so as the strategies are written now it is exactly
  `sqrt(CTX share of the tile x CRISM share of the tile)`. SHARAD is `timeless`,
  so it is asked of the whole record rather than of the window and never enters
  the score. The days a window runs are priced against the ground it reaches
  while the search is choosing the window, but that charge is not carried into
  the number reported here.

The score reads about the same under every strategy, and that is the honest
answer rather than a bug. All four strategies ask for the same `constraints`, so
all four share the same floor, `sqrt(0.50 x 0.30)` = 38.7%, and the score is only
ever measured on tiles that had already cleared it. Neither `tile_km` nor
`span_days` enters the score at all. What separates the strategies is how many
tiles clear that floor, which is the table below.

In [ ]:
"""Tabulate how long a window runs and how far it reaches."""

coverage.windows(read)

## Final dataset size

What each strategy would hand over: the tiles it keeps of the tiles it
searched, how many of them hold more than one instrument, and how many of
them two instruments really overlap on rather than merely sharing a tile.

### The dataset each strategy would leave

- **Tiles searched** is every tile holding anything measurable.
- **Tiles kept** is those that earned a window worth keeping, so the tiles the
  dataset would actually be built from.
- **Share kept** is the second over the first, and is the number that really
  separates one strategy from another.

In [ ]:
"""Tabulate the dataset each strategy would leave behind."""

size.final(read)

## The dataset that was measured

How much there is to choose from, before any strategy is asked of it. Every
number here is read off the measurement itself, so it is the same whichever
strategy the rest of the notebook goes on to compare.

### The measurement at a glance

Read straight off the coverage measurement, so it is the same whichever strategy
the rest of the notebook compares.

- **Features** is how many the coverage stage measured.
- **Feature classes** is how many kinds the catalogue files them under.
- **Features dropped as points** is how many the catalogue gives no extent, so
  there was no ground to crop an observation to and nothing was downloaded.
- **Ground** is what their bounding boxes hold between them.

In [ ]:
"""Read the catalogue index, which every table below is drawn from."""

measured = catalogue.read()
overall.measured(measured)

### How many features of each class each strategy would select

**Features measured** is what the coverage stage holds of that class. Each
strategy column counts the features of that class it would select, meaning at
least one tile of the feature earned a window worth keeping, however many of its
other tiles were refused. It says which kinds of landform survive a strategy and
which it wipes out entirely.

In [ ]:
"""Tabulate how many features of each class each strategy would select."""

overall.classes(measured, read)

### What each instrument holds

One row per instrument, over the whole measured record and before any strategy.

- **Features reached** is how many features it took any observation of.
- **Observations** is how many it took of them altogether.
- **Ground reached** and **Share of the ground** are how much of the features'
  ground it covered, counting a cell once however often it was revisited.
- **First look** and **Last look** bound the record it offers a window.

In [ ]:
"""Tabulate what each instrument holds of the measured features."""

overall.instruments(measured)

### Median share of a feature reached, against how big it is

Whether an instrument stops covering a feature once the feature gets big, which
is what decides how wide a tile can usefully be cut.

The horizontal axis is the ground a feature's bounding box holds, on a log scale.
The vertical axis is the share of that ground the instrument reached over its
whole record. Every feature is placed in one of 18 log-spaced size bands, and one
point is drawn per band per instrument at the **median** of the features in it,
skipping any band holding fewer than five. It is a median rather than a mean so
that a few fully covered outliers cannot lift a band.

In [ ]:
"""Draw how much of a feature each instrument reached against its size."""

reach.against_size(measured)